In [ ]:
!pip -q install "transformers>=4.45,<5" "datasets>=3,<4" accelerate sentencepiece rouge-score pypdf gradio

In [ ]:
import os
import gc
import json
import tarfile
import time
import shutil
from pathlib import Path

import torch
import pandas as pd

from huggingface_hub import hf_hub_download
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: GPU is unavailable. Select a GPU runtime in Colab.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs("/content/summarizer_models", exist_ok=True)

In [ ]:
LANGUAGES = [
    "english",
    "hindi",
    "telugu",
    "tamil",
    "bengali",
    "marathi",
    "gujarati",
    "punjabi"
]

LANGUAGE_LABELS = {
    "english": "English",
    "hindi": "Hindi",
    "telugu": "Telugu",
    "tamil": "Tamil",
    "bengali": "Bengali",
    "marathi": "Marathi",
    "gujarati": "Gujarati",
    "punjabi": "Punjabi"
}

print("Languages:", LANGUAGES)
print("Total languages:", len(LANGUAGES))

In [ ]:
for lang in LANGUAGES:
    download_and_extract_language(lang)

print("All eight language datasets are ready.")

In [ ]:
raw_data = {}

for lang in LANGUAGES:
    language_dir = DATA_ROOT / lang

    files = {
        "train": list(language_dir.rglob(f"{lang}_train.jsonl")),
        "validation": list(language_dir.rglob(f"{lang}_val.jsonl")),
        "test": list(language_dir.rglob(f"{lang}_test.jsonl"))
    }

    for split, matches in files.items():
        if not matches:
            raise FileNotFoundError(
                f"Missing {split} JSONL file for {lang} in {language_dir}"
            )

    raw_data[lang] = DatasetDict({
        split: load_dataset(
            "json",
            data_files=str(files[split][0]),
            split="train"
        )
        for split in ["train", "validation", "test"]
    })

    print(
        lang,
        "train =", len(raw_data[lang]["train"]),
        "validation =", len(raw_data[lang]["validation"]),
        "test =", len(raw_data[lang]["test"])
    )

print("All language datasets loaded.")

In [ ]:
for lang in LANGUAGES:
    example = raw_data[lang]["train"][0]

    print("\nLanguage:", lang)
    print("Columns:", list(example.keys()))
    print("Article sample:", str(example.get("text", ""))[:250])
    print("Summary sample:", str(example.get("summary", ""))[:150])

    if not example.get("text") or not example.get("summary"):
        raise ValueError(f"Empty article or summary found for {lang}")

print("\nDataset inspection completed.")

In [ ]:
TRAIN_PER_LANG = 200
VAL_PER_LANG = 20
TEST_PER_LANG = 20

split_limits = {
    "train": TRAIN_PER_LANG,
    "validation": VAL_PER_LANG,
    "test": TEST_PER_LANG
}

balanced_parts = {
    "train": [],
    "validation": [],
    "test": []
}

for lang in LANGUAGES:
    for split, limit in split_limits.items():
        ds = raw_data[lang][split]

        # Remove empty records
        ds = ds.filter(
            lambda row: (
                row.get("text") is not None
                and row.get("summary") is not None
                and len(str(row["text"]).strip()) > 0
                and len(str(row["summary"]).strip()) > 0
            )
        )

        if len(ds) < limit:
            raise ValueError(
                f"{lang} has only {len(ds)} usable records in {split}; "
                f"{limit} were requested."
            )

        ds = ds.shuffle(seed=SEED).select(range(limit))

        # Explicitly store the language for every record
        ds = ds.map(
            lambda row: {"language": lang}
        )

        balanced_parts[split].append(ds)

balanced_data = DatasetDict({
    split: concatenate_datasets(parts).shuffle(seed=SEED)
    for split, parts in balanced_parts.items()
})

for split, ds in balanced_data.items():
    print(split, len(ds))

print("Training language counts:")
print(pd.Series(balanced_data["train"]["language"]).value_counts())

In [ ]:
MODEL_IDS = {
    "mt5-small": "google/mt5-small",
    "umt5-small": "google/umt5-small",
    "byt5-small": "google/byt5-small"
}

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 96

print("Models to compare:")
for name, model_id in MODEL_IDS.items():
    print(name, "->", model_id)

In [ ]:
def prepare_tokenized_data(tokenizer):
    result = {}

    for split in ["train", "validation", "test"]:
        def preprocess(batch):
            inputs = [
                f"summarize in {lang}: {article}"
                for article, lang in zip(
                    batch["text"], batch["language"]
                )
            ]

            model_inputs = tokenizer(
                inputs,
                max_length=MAX_SOURCE_LENGTH,
                truncation=True
            )

            targets = tokenizer(
                text_target=batch["summary"],
                max_length=MAX_TARGET_LENGTH,
                truncation=True
            )

            model_inputs["labels"] = targets["input_ids"]
            return model_inputs

        result[split] = balanced_data[split].map(
            preprocess,
            batched=True,
            remove_columns=balanced_data[split].column_names
        )

    return DatasetDict(result)

tokenized_by_model = {}

for model_name, model_id in MODEL_IDS.items():
    print("Tokenizing:", model_name)

    tok = AutoTokenizer.from_pretrained(model_id)
    tokenized_by_model[model_name] = prepare_tokenized_data(tok)

    print({
        split: len(ds)
        for split, ds in tokenized_by_model[model_name].items()
    })

    del tok
    gc.collect()

print("Tokenization completed.")

In [ ]:
import math

TRAINING_STEPS = 100
OUTPUT_ROOT = Path("/content/summarizer_models")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

training_results = {}

for model_name, model_id in MODEL_IDS.items():
    print("\n" + "=" * 60)
    print("Training:", model_name)
    print("=" * 60)

    start_time = time.time()

    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(DEVICE)

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    collator = DataCollatorForSeq2Seq(
        tokenizer=tok,
        model=model
    )

    args = Seq2SeqTrainingArguments(
        output_dir=f"/content/{model_name}_training",
        max_steps=TRAINING_STEPS,
        learning_rate=1e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        optim="adafactor",
        fp16=False,
        bf16=False,
        gradient_checkpointing=True,
        max_grad_norm=1.0,
        eval_strategy="steps",
        eval_steps=50,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=0
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_by_model[model_name]["train"],
        eval_dataset=tokenized_by_model[model_name]["validation"],
        data_collator=collator,
        processing_class=tok
    )

    train_output = trainer.train()
    eval_output = trainer.evaluate()

    train_loss = train_output.training_loss
    validation_loss = eval_output["eval_loss"]

    print("Final training loss:", train_loss)
    print("Final validation loss:", validation_loss)

    if not math.isfinite(train_loss):
        raise RuntimeError(
            f"{model_name}: training loss is invalid. Do not use this model."
        )

    if not math.isfinite(validation_loss):
        raise RuntimeError(
            f"{model_name}: validation loss is invalid. Do not use this model."
        )

    model.config.use_cache = True

    model_dir = OUTPUT_ROOT / model_name
    trainer.save_model(str(model_dir))
    tok.save_pretrained(str(model_dir))

    elapsed = time.time() - start_time

    training_results[model_name] = {
        "model_id": model_id,
        "training_loss": float(train_loss),
        "validation_loss": float(validation_loss),
        "training_seconds": round(elapsed, 2),
        "training_steps": TRAINING_STEPS,
        "model_directory": str(model_dir)
    }

    with open(OUTPUT_ROOT / "training_results.json", "w") as f:
        json.dump(training_results, f, indent=2)

    print("Saved to:", model_dir)
    print("Training time in minutes:", round(elapsed / 60, 2))

    del trainer, model, tok, collator, args
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTraining completed for all models.")
print(json.dumps(training_results, indent=2))

In [ ]:
from rouge_score import rouge_scorer

rouge_metric = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False
)

evaluation_rows = []
prediction_rows = []

for model_name in MODEL_IDS:
    print("\nEvaluating:", model_name)

    model_dir = OUTPUT_ROOT / model_name

    tok = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForSeq2SeqLM.from_pretrained(
        str(model_dir)
    ).to(DEVICE)

    model.eval()
    model.config.use_cache = True

    for example in balanced_data["test"]:
        lang = example["language"]
        article = example["text"]
        reference = example["summary"]

        prompt = f"summarize in {lang}: {article}"

        inputs = tok(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SOURCE_LENGTH
        ).to(DEVICE)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=2,
                early_stopping=True
            )

        prediction = tok.decode(
            generated[0],
            skip_special_tokens=True
        ).strip()

        scores = rouge_metric.score(reference, prediction)

        row = {
            "model": model_name,
            "language": lang,
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure
        }

        evaluation_rows.append(row)

        prediction_rows.append({
            "model": model_name,
            "language": lang,
            "article": article,
            "reference_summary": reference,
            "generated_summary": prediction,
            "rouge1": row["rouge1"],
            "rouge2": row["rouge2"],
            "rougeL": row["rougeL"]
        })

    del model, tok
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Finished:", model_name)

evaluation_df = pd.DataFrame(evaluation_rows)
predictions_df = pd.DataFrame(prediction_rows)

evaluation_df.to_csv(
    "/content/summarizer_models/evaluation_metrics.csv",
    index=False
)

predictions_df.to_csv(
    "/content/summarizer_models/test_predictions.csv",
    index=False
)

print("Evaluation completed.")
print("Prediction records:", len(predictions_df))

In [ ]:
rouge_summary = (
    evaluation_df
    .groupby("model")[["rouge1", "rouge2", "rougeL"]]
    .mean()
    .reset_index()
)

training_df = pd.DataFrame([
    {
        "model": name,
        "training_loss": result["training_loss"],
        "validation_loss": result["validation_loss"],
        "training_minutes": result["training_seconds"] / 60
    }
    for name, result in training_results.items()
])

comparison_df = rouge_summary.merge(
    training_df,
    on="model",
    how="left"
)

comparison_df = comparison_df.sort_values(
    "rougeL",
    ascending=False
).reset_index(drop=True)

comparison_df.to_csv(
    "/content/summarizer_models/model_comparison.csv",
    index=False
)

print("OVERALL MODEL COMPARISON")
display(comparison_df.round(4))

print("\nROUGE-L BY LANGUAGE")
language_comparison = (
    evaluation_df
    .groupby(["model", "language"])["rougeL"]
    .mean()
    .reset_index()
)

display(
    language_comparison.pivot(
        index="language",
        columns="model",
        values="rougeL"
    ).round(4)
)

BEST_MODEL = comparison_df.iloc[0]["model"]

print("\nBest model by average ROUGE-L:", BEST_MODEL)

In [ ]:

import re
import torch

TELUGU_SCRIPT = re.compile(r"[\u0C00-\u0C7F]")
HINDI_SCRIPT = re.compile(r"[\u0900-\u097F]")
TAMIL_SCRIPT = re.compile(r"[\u0B80-\u0BFF]")
BENGALI_SCRIPT = re.compile(r"[\u0980-\u09FF]")
GURMUKHI_SCRIPT = re.compile(r"[\u0A00-\u0A7F]")
GUJARATI_SCRIPT = re.compile(r"[\u0A80-\u0AFF]")
DEVANAGARI_SCRIPT = HINDI_SCRIPT

LANGUAGE_SCRIPTS = {
    "Telugu": TELUGU_SCRIPT,
    "Hindi": HINDI_SCRIPT,
    "Tamil": TAMIL_SCRIPT,
    "Bengali": BENGALI_SCRIPT,
    "Punjabi": GURMUKHI_SCRIPT,
    "Gujarati": GUJARATI_SCRIPT,
    "Marathi": DEVANAGARI_SCRIPT,
}

def contains_expected_script(text, language):
    pattern = LANGUAGE_SCRIPTS.get(language)
    if language == "English":
        return bool(re.search(r"[A-Za-z]", text))
    if pattern is None:
        return True
    return bool(pattern.search(text))


def summarize_pdf(pdf_file, language_label, model_name):
    if not pdf_file:
        return "Please upload a PDF file."

    try:
        model, tokenizer = load_selected_model(model_name)
        document_text = extract_pdf_text(pdf_file)

        if len(document_text.strip()) < 100:
            return "The PDF contains too little readable text to summarize."

        # Use smaller chunks to reduce truncation.
        chunks = split_into_chunks(
            document_text,
            tokenizer,
            chunk_tokens=350,
            overlap=30
        )

        if not chunks:
            return "No readable text was found in the PDF."

        # Ask for one concise factual point from each chunk.
        point_prompt = {
            "Telugu": "Write one important factual summary point in natural Telugu script.",
            "Hindi": "Write one important factual summary point in Hindi.",
            "Tamil": "Write one important factual summary point in Tamil script.",
            "Bengali": "Write one important factual summary point in Bengali script.",
            "Marathi": "Write one important factual summary point in Marathi.",
            "Gujarati": "Write one important factual summary point in Gujarati script.",
            "Punjabi": "Write one important factual summary point in Punjabi script.",
            "English": "Write one important factual summary point in English."
        }.get(language_label, "Write one important factual summary point.")

        points = []
        seen = set()

        # Limit work for very large PDFs.
        # Select chunks spread across the document.
        max_chunks = min(len(chunks), 20)
        if len(chunks) > max_chunks:
            indexes = [
                round(i * (len(chunks) - 1) / (max_chunks - 1))
                for i in range(max_chunks)
            ]
            selected_chunks = [chunks[i] for i in indexes]
        else:
            selected_chunks = chunks

        for chunk in selected_chunks:
            prompt = (
                f"Summarize in {language_label}. "
                f"{point_prompt} "
                "Use only information in the text. "
                "Do not repeat the text verbatim.\n\n"
                f"Text:\n{chunk}"
            )

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(model.device)

            with torch.no_grad():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    num_beams=4,
                    no_repeat_ngram_size=3,
                    repetition_penalty=1.2,
                    early_stopping=True
                )

            point = tokenizer.decode(
                generated[0],
                skip_special_tokens=True
            ).strip()

            point = re.sub(r"^\s*[-*\d.)]+\s*", "", point).strip()
            normalized = re.sub(r"\s+", " ", point).lower()

            if len(point) < 8 or normalized in seen:
                continue

            seen.add(normalized)
            points.append(point)

            # Stop once we have ten distinct candidate points.
            if len(points) >= 10:
                break

        if len(points) < 10:
            return (
                f"Only {len(points)} distinct points could be generated. "
                "This model may need further fine-tuning. "
                "Try a smaller PDF or improve the model before relying "
                "on its summaries."
            )

        # Check language before presenting the result as Telugu, Hindi, etc.
        if not all(
            contains_expected_script(point, language_label)
            for point in points
        ):
            return (
                f"The model failed the {language_label} language check. "
                "It generated text in another language. "
                "The model needs better multilingual fine-tuning; "
                "the output has not been presented as a valid translation."
            )

        return (
            f"Model: {model_name}\n"
            f"Output language: {language_label}\n"
            f"Number of points: {len(points)}\n\n"
            "SUMMARY\n\n"
            + "\n".join(
                f"{i}. {point}"
                for i, point in enumerate(points, 1)
            )
        )

    except Exception as exc:
        return f"Could not summarize the PDF: {exc}"


In [ ]:
!pip -q install "transformers>=4.45,<5" accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PDF_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

pdf_tokenizer = AutoTokenizer.from_pretrained(PDF_MODEL_ID)

pdf_model = AutoModelForCausalLM.from_pretrained(
    PDF_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("PDF summarization model loaded.")

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(
    name for name in uploaded
    if name.lower().endswith(".pdf")
)

print("PDF uploaded successfully:", pdf_path)

In [ ]:
!pip install -q pypdf
from pypdf import PdfReader

reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))

for i, page in enumerate(reader.pages[:3]):
    text = page.extract_text() or ""

    print(f"\n--- PAGE {i + 1} ---")
    print(text[:1500])

In [ ]:
import re
from collections import Counter

print("PDF file:", pdf_path)

# Initialize and populate the 'pages' variable from our reader object
pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text_content = page.extract_text() or ""
    text_content = re.sub(r"[ \t]+", " ", text_content)
    text_content = re.sub(r"\n{3,}", "\n\n", text_content)
    if text_content.strip():
        pages.append({
            "page": page_number,
            "text": text_content.strip()
        })

print("Pages extracted:", len(pages))

for item in pages[:5]:
    print(f"\n--- PDF PAGE {item['page']} ---")
    print(item["text"][:700])

all_pdf_text = "\n".join(item["text"] for item in pages)

print("\nTotal extracted characters:", len(all_pdf_text))
print("Contains 'autoencoder':", "autoencoder" in all_pdf_text.lower())
print("Contains 'hadoop':", "hadoop" in all_pdf_text.lower())

In [ ]:
!pip -q install "transformers>=4.45,<5" accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PDF_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

pdf_tokenizer = AutoTokenizer.from_pretrained(PDF_MODEL_ID)

pdf_model = AutoModelForCausalLM.from_pretrained(
    PDF_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("PDF summarization model loaded.")

In [ ]:
def generate_theory(prompt, max_new_tokens=400):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an academic textbook summarizer. "
                "Use only facts from the supplied source. "
                "Explain concepts in connected, meaningful paragraphs. "
                "Never invent topics or replace the source with generic "
                "study advice. Preserve technical terminology."
            )
        },
        {"role": "user", "content": prompt}
    ]

    formatted = pdf_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = pdf_tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=5000
    ).to(pdf_model.device)

    with torch.no_grad():
        outputs = pdf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return pdf_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

In [ ]:
def make_chunks(text, max_chars=5000):
    paragraphs = [
        p.strip() for p in re.split(r"\n\s*\n", text)
        if len(p.strip()) > 30
    ]

    chunks = []
    current = ""

    for paragraph in paragraphs:
        if len(current) + len(paragraph) > max_chars:
            if current:
                chunks.append(current)
            current = paragraph
        else:
            current = f"{current}\n\n{paragraph}".strip()

    if current:
        chunks.append(current)

    return chunks


source_chunks = make_chunks(all_pdf_text)

print("Source chunks:", len(source_chunks))
print("\nFirst chunk preview:\n", source_chunks[0][:1200])

In [ ]:
theory_parts = []

for index, chunk in enumerate(source_chunks, start=1):
    prompt = f"""
Write a theory-style explanation of the following textbook content.

Requirements:
- Use one or more connected paragraphs.
- Explain the actual definitions, concepts, working principles,
  and examples found in the source.
- Keep important technical terms.
- Do not add generic instructions such as "understand this topic".
- Do not invent concepts absent from the source.
- Do not create a page-by-page summary.
- Preserve the source language (English) for this first pass.

SOURCE TEXT:
{chunk}
"""

    explanation = generate_theory(prompt, max_new_tokens=400)

    theory_parts.append(explanation)
    print(f"Processed chunk {index}/{len(source_chunks)}")